<a href="https://colab.research.google.com/github/karishmaram-tech/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karishmaram-tech/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### My baseline rule

I will prioritize content where search position suggests an opportunity and where the content still has measurable search visibility.

- Position opportunity = average February Google Search Console position is 10–50.
- Stronger opportunity = average position is 20–50.
- Search visibility = at least 100 February Google Search Console impressions.
- Higher score means higher priority for manual review or refresh consideration.

### Reason codes

- POSITION_20_50_VISIBLE — position is 20–50 and impressions are at least 100.
- POSITION_10_20_VISIBLE — position is 10–20 and impressions are at least 100.
- POSITION_20_50_LOW_VISIBILITY — position is 20–50 but impressions are below 100.
- OTHER — does not match the main opportunity signals.

### Action labels

- REFRESH_REVIEW — strongest baseline opportunity signal.
- REVIEW — secondary opportunity signal.
- MONITOR — no strong baseline opportunity signal.

In [2]:
# ==========================================
# W04 — SECTION 1
# Load February data and content metadata
# ==========================================

import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata

# ------------------------------------------
# Hugging Face connection
# ------------------------------------------

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf
    (TYPE huggingface, TOKEN '{HF_TOKEN}')
    """
)

REL = "hf://datasets/FlyRank/internship-warehouse"

FACT = f"{REL}/fact_content_daily_performance"

# Decision point
DECISION_DATE = pd.Timestamp("2026-02-28")

print("Connection ready.")
print("Decision date:", DECISION_DATE.date())

Connection ready.
Decision date: 2026-02-28


In [3]:
# ==========================================
# Load February 2026 fact data
# ==========================================

feb_fact = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS feb_impressions,
        SUM(gsc_clicks) AS feb_clicks,
        AVG(gsc_avg_position) AS feb_avg_position
    FROM read_parquet(
        '{FACT}/month=2026-02/*.parquet'
    )
    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

print("February fact rows:", len(feb_fact))
print("Unique clients:", feb_fact["client_hash_id"].nunique())
print("Unique content items:", feb_fact["content_hash_id"].nunique())

display(feb_fact.head())

February fact rows: 321546
Unique clients: 54
Unique content items: 321546


,client_hash_id,content_hash_id,feb_impressions,feb_clicks,feb_avg_position
0,client_e547b89c05043229,content_1eea820697c3b95a,299.0,0.0,12.946228
1,client_e547b89c05043229,content_9abd8b303f805847,733.0,6.0,6.495085
2,client_e547b89c05043229,content_5f58c55cbfee172a,514.0,0.0,10.490023
3,client_e547b89c05043229,content_6fe390ba3af1e456,2931.0,3.0,38.436254
4,client_e547b89c05043229,content_3ad5d2160242b9ca,970.0,2.0,9.710810


In [6]:
# ==========================================
# Load content metadata
# ==========================================

content = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        last_optimized_date,
        content_updated_date,
        content_type,
        search_volume,
        word_count,
        is_published,
        is_deleted
    FROM read_parquet(
        '{REL}/dim_content.parquet'
    )
""").df()

print("Content rows:", len(content))
print("Content columns:")
print(content.columns.tolist())

display(content.head())

Content rows: 519606
Content columns:
['client_hash_id', 'content_hash_id', 'last_optimized_date', 'content_updated_date', 'content_type', 'search_volume', 'word_count', 'is_published', 'is_deleted']


,client_hash_id,content_hash_id,last_optimized_date,content_updated_date,content_type,search_volume,word_count,is_published,is_deleted
0,client_04660893ae39614a,content_004de9653278b5a4,NaT,2026-07-01,keyword article,30,2555,True,False
1,client_04660893ae39614a,content_00dc5efae381b2ab,NaT,2026-07-01,keyword article,10,2430,True,False
2,client_04660893ae39614a,content_01410f2556c327ac,NaT,2026-07-01,keyword article,480,2645,True,False
3,client_04660893ae39614a,content_019f27f634053ca7,NaT,2026-06-15,keyword article,0,2522,True,False
4,client_04660893ae39614a,content_01efa71faea45dcc,NaT,2026-06-01,keyword article,2400,2552,True,False


In [7]:
# ==========================================
# Merge February performance + content data
# ==========================================

df = feb_fact.merge(
    content,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

# Convert optimization date
df["last_optimized_date"] = pd.to_datetime(
    df["last_optimized_date"],
    errors="coerce"
)

# Calculate days since last optimization
df["days_since_optimized"] = (
    DECISION_DATE - df["last_optimized_date"]
).dt.days

# Keep published, non-deleted content
df = df[
    (df["is_deleted"] != True) &
    (df["is_published"] != False)
].copy()

print("Rows after content filters:", len(df))
print(
    "Rows with optimization date:",
    df["days_since_optimized"].notna().sum()
)

display(
    df[
        [
            "client_hash_id",
            "content_hash_id",
            "last_optimized_date",
            "days_since_optimized",
            "feb_impressions",
            "search_volume"
        ]
    ].head(10)
)

Rows after content filters: 301152
Rows with optimization date: 39820


,client_hash_id,content_hash_id,last_optimized_date,days_since_optimized,feb_impressions,search_volume
0,client_e547b89c05043229,content_1eea820697c3b95a,2026-05-27,-88.0,299.0,140
1,client_e547b89c05043229,content_9abd8b303f805847,2026-06-22,-114.0,733.0,90
2,client_e547b89c05043229,content_5f58c55cbfee172a,2026-05-27,-88.0,514.0,720
3,client_e547b89c05043229,content_6fe390ba3af1e456,2026-05-26,-87.0,2931.0,110
4,client_e547b89c05043229,content_3ad5d2160242b9ca,2026-05-26,-87.0,970.0,70
5,client_e547b89c05043229,content_a2bd730a7cf68316,2026-05-26,-87.0,551.0,50
6,client_e547b89c05043229,content_cbe43d4b6ce2d320,2026-05-27,-88.0,291.0,50
7,client_e547b89c05043229,content_babd931911c9ee33,2026-07-02,-124.0,2680.0,20
8,client_e547b89c05043229,content_9c36ace83c73b5eb,2026-05-27,-88.0,416.0,50
9,client_e547b89c05043229,content_431784c057b25a5d,2026-05-15,-76.0,3641.0,20


In [8]:
# ==========================================
# SIGNAL 1 — STALENESS
# ==========================================

df["staleness_bucket"] = pd.cut(
    df["days_since_optimized"],
    bins=[-np.inf, 89, 179, 364, np.inf],
    labels=[
        "<90 days",
        "90–179 days",
        "180–364 days",
        "365+ days"
    ]
)

staleness_check = (
    df.groupby(
        "staleness_bucket",
        observed=False
    )
    .agg(
        n=("content_hash_id", "size"),
        median_impressions=("feb_impressions", "median"),
        mean_impressions=("feb_impressions", "mean")
    )
    .reset_index()
)

print("SIGNAL 1 — STALENESS")
display(staleness_check)

SIGNAL 1 — STALENESS


,staleness_bucket,n,median_impressions,mean_impressions
0,<90 days,39820,1255.5,3210.583074
1,90–179 days,0,NaN,NaN
2,180–364 days,0,NaN,NaN
3,365+ days,0,NaN,NaN


In [9]:
# ==========================================
# SIGNAL 2 — SEARCH VISIBILITY
# ==========================================

df["visibility_bucket"] = pd.cut(
    df["feb_impressions"],
    bins=[-np.inf, 99, 499, 999, np.inf],
    labels=[
        "<100",
        "100–499",
        "500–999",
        "1000+"
    ]
)

visibility_check = (
    df.groupby(
        "visibility_bucket",
        observed=False
    )
    .agg(
        n=("content_hash_id", "size"),
        median_days_since_optimized=("days_since_optimized", "median"),
        mean_days_since_optimized=("days_since_optimized", "mean")
    )
    .reset_index()
)

print("SIGNAL 2 — SEARCH VISIBILITY")
display(visibility_check)

SIGNAL 2 — SEARCH VISIBILITY


,visibility_bucket,n,median_days_since_optimized,mean_days_since_optimized
0,<100,221154,-90.0,-98.157223
1,100–499,32913,-90.0,-96.211405
2,500–999,13801,-88.0,-98.102935
3,1000+,33284,-103.0,-102.516151


In [10]:
# ==========================================
# SIGNAL 3 — SEARCH POSITION
# ==========================================

df["position_bucket"] = pd.cut(
    df["feb_avg_position"],
    bins=[-np.inf, 5, 10, 20, 50, np.inf],
    labels=[
        "Top 5",
        "5–10",
        "10–20",
        "20–50",
        "50+"
    ]
)

position_check = (
    df.groupby(
        "position_bucket",
        observed=False
    )
    .agg(
        n=("content_hash_id", "size"),
        median_impressions=("feb_impressions", "median"),
        median_position=("feb_avg_position", "median")
    )
    .reset_index()
)

print("SIGNAL 3 — SEARCH POSITION")
display(position_check)

SIGNAL 3 — SEARCH POSITION


,position_bucket,n,median_impressions,median_position
0,Top 5,42098,308.0,3.224986
1,5–10,51586,107.0,7.222861
2,10–20,30805,128.0,13.453704
3,20–50,21325,91.0,28.679825
4,50+,4763,27.0,61.656878


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ==========================================
# SECTION 2 — BUILD RANKED QUEUE
# ==========================================

import os # Import the os module

queue = df.copy()

# ------------------------------------------
# Signal 1: Search-position opportunity
# ------------------------------------------

queue["position_20_50_signal"] = (
    (queue["feb_avg_position"] >= 20) &
    (queue["feb_avg_position"] < 50)
).astype(int)

queue["position_10_20_signal"] = (
    (queue["feb_avg_position"] >= 10) &
    (queue["feb_avg_position"] < 20)
).astype(int)

# ------------------------------------------
# Signal 2: Search visibility
# ------------------------------------------

queue["visibility_signal"] = (
    queue["feb_impressions"] >= 100
).astype(int)

# ------------------------------------------
# Score
# ------------------------------------------

queue["score"] = (
    queue["position_20_50_signal"] * 3 +
    queue["position_10_20_signal"] * 2 +
    queue["visibility_signal"]
)

# ------------------------------------------
# Reason code
# ------------------------------------------

def assign_reason(row):

    if (
        row["position_20_50_signal"] == 1
        and row["visibility_signal"] == 1
    ):
        return "POSITION_20_50_VISIBLE"

    elif (
        row["position_10_20_signal"] == 1
        and row["visibility_signal"] == 1
    ):
        return "POSITION_10_20_VISIBLE"

    elif row["position_20_50_signal"] == 1:
        return "POSITION_20_50_LOW_VISIBILITY"

    else:
        return "OTHER"


queue["reason_code"] = queue.apply(
    assign_reason,
    axis=1
)

# ------------------------------------------
# Action
# ------------------------------------------

def assign_action(row):

    if row["reason_code"] == "POSITION_20_50_VISIBLE":
        return "REFRESH_REVIEW"

    elif row["reason_code"] == "POSITION_10_20_VISIBLE":
        return "REVIEW"

    elif row["reason_code"] == "POSITION_20_50_LOW_VISIBILITY":
        return "REVIEW"

    else:
        return "MONITOR"


queue["action"] = queue.apply(
    assign_action,
    axis=1
)

# ------------------------------------------
# Rank
# ------------------------------------------

queue = queue.sort_values(
    [
        "score",
        "feb_impressions"
    ],
    ascending=[
        False,
        False
    ],
    na_position="last"
).reset_index(drop=True)

queue["rank"] = np.arange(
    1,
    len(queue) + 1
)

# ------------------------------------------
# Output
# ------------------------------------------

output_columns = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "score",
    "action",
    "reason_code",
    "feb_impressions",
    "feb_avg_position",
    "search_volume",
    "content_type"
]

baseline_queue = queue[
    output_columns
].copy()

# Create the directory if it doesn't exist
output_dir = "work/outputs"
os.makedirs(output_dir, exist_ok=True)

baseline_queue.to_csv(
    os.path.join(output_dir, "baseline_action_score.csv"), # Use os.path.join for path construction
    index=False
)

print("Queue rows:", len(baseline_queue))
print(f"Saved to: {os.path.join(output_dir, 'baseline_action_score.csv')}") # Update print statement

display(
    baseline_queue.head(20)
)

Queue rows: 301152
Saved to: work/outputs/baseline_action_score.csv


,rank,client_hash_id,content_hash_id,score,action,reason_code,feb_impressions,feb_avg_position,search_volume,content_type
0,1,client_23a62021009f63c4,content_36e53e9c707674fc,4,REFRESH_REVIEW,POSITION_20_50_VISIBLE,100736.0,34.040988,0,keyword article
1,2,client_fef1a8f436438636,content_84a6bf3578312e90,4,REFRESH_REVIEW,POSITION_20_50_VISIBLE,79986.0,20.050958,390,keyword article
2,3,client_fef1a8f436438636,content_ba462518dad435fc,4,REFRESH_REVIEW,POSITION_20_50_VISIBLE,69134.0,27.753576,0,keyword article
3,4,client_23a62021009f63c4,content_3df3f32f3fd58dea,4,REFRESH_REVIEW,POSITION_20_50_VISIBLE,68216.0,24.899933,10,keyword article
4,5,client_23a62021009f63c4,content_b51957d7f4abe47e,4,REFRESH_REVIEW,POSITION_20_50_VISIBLE,57558.0,27.816678,0,keyword article
5,6,client_23a62021009f63c4,content_bdf60c86117079be,4,REFRESH_REVIEW,POSITION_20_50_VISIBLE,51346.0,33.418919,10,keyword article
6,7,client_fef1a8f436438636,content_0aaa197051f58d6f,4,REFRESH_REVIEW,POSITION_20_50_VISIBLE,49903.0,34.694009,10,keyword article
7,8,client_23a62021009f63c4,content_a3a1317f7c2bc3dd,4,REFRESH_REVIEW,POSITION_20_50_VISIBLE,47036.0,29.355751,0,keyword article
8,9,client_23a62021009f63c4,content_87b9c790d43001dc,4,REFRESH_REVIEW,POSITION_20_50_VISIBLE,46002.0,30.531879,0,keyword article
9,10,client_23a62021009f63c4,content_a3b985d3ee8a219b,4,REFRESH_REVIEW,POSITION_20_50_VISIBLE,45744.0,24.343509,0,keyword article


In [13]:
print("ACTION DISTRIBUTION")

display(
    baseline_queue["action"]
    .value_counts()
    .rename_axis("action")
    .reset_index(name="n")
)

print("\nREASON CODE DISTRIBUTION")

display(
    baseline_queue["reason_code"]
    .value_counts()
    .rename_axis("reason_code")
    .reset_index(name="n")
)

ACTION DISTRIBUTION


,action,n
0,MONITOR,263147
1,REVIEW,27687
2,REFRESH_REVIEW,10318



REASON CODE DISTRIBUTION


,reason_code,n
0,OTHER,263147
1,POSITION_10_20_VISIBLE,16645
2,POSITION_20_50_LOW_VISIBILITY,11042
3,POSITION_20_50_VISIBLE,10318


## Top-20 Review

The top 20 records are reviewed manually to check whether the baseline rule produces a sensible prioritization.

For each item, I record:
- the action assigned by the rule
- the reason code behind the score
- a confidence note based only on February signals
- what additional evidence could make the rule wrong

This is a baseline prioritization, not a prediction of future performance.

### Top-20 Review Summary

The top 20 items are all assigned `REFRESH_REVIEW` because they satisfy the strongest baseline condition: average February search position between 20 and 50 together with at least 100 impressions.

The ranking is further ordered by February impressions, so items with the same score and stronger observed visibility appear earlier.

The main limitation is that the baseline uses only a small number of February signals. It does not include future March outcomes, labels, or private client information.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Top-20 review table

top20_review = baseline_queue.head(20).copy()

def confidence_note(row):
    if row["reason_code"] == "POSITION_20_50_VISIBLE":
        return "Medium — visible impressions with position 20–50."
    elif row["reason_code"] == "POSITION_10_20_VISIBLE":
        return "Medium — visible impressions with position 10–20."
    elif row["reason_code"] == "POSITION_20_50_LOW_VISIBILITY":
        return "Low — position signal exists, but visibility is limited."
    else:
        return "Low — does not match the main opportunity signals."

def wrong_if(row):
    if row["reason_code"] == "POSITION_20_50_VISIBLE":
        return "Could be wrong if the position reflects a temporary query mix or the content is already performing well for its target intent."
    elif row["reason_code"] == "POSITION_10_20_VISIBLE":
        return "Could be wrong if the content is intentionally maintained in this position or has limited improvement potential."
    elif row["reason_code"] == "POSITION_20_50_LOW_VISIBILITY":
        return "Could be wrong if low impressions indicate very limited demand rather than an optimization opportunity."
    else:
        return "Could be wrong if another important signal is missing from the baseline."

top20_review["confidence_note"] = top20_review.apply(confidence_note, axis=1)
top20_review["what_would_make_wrong"] = top20_review.apply(wrong_if, axis=1)

review_columns = [
    "rank",
    "content_hash_id",
    "action",
    "reason_code",
    "feb_impressions",
    "feb_avg_position",
    "search_volume",
    "confidence_note",
    "what_would_make_wrong"
]

display(top20_review[review_columns])

,rank,content_hash_id,action,reason_code,feb_impressions,feb_avg_position,search_volume,confidence_note,what_would_make_wrong
0,1,content_36e53e9c707674fc,REFRESH_REVIEW,POSITION_20_50_VISIBLE,100736.0,34.040988,0,Medium — visible impressions with position 20–50.,Could be wrong if the position reflects a temp...
1,2,content_84a6bf3578312e90,REFRESH_REVIEW,POSITION_20_50_VISIBLE,79986.0,20.050958,390,Medium — visible impressions with position 20–50.,Could be wrong if the position reflects a temp...
2,3,content_ba462518dad435fc,REFRESH_REVIEW,POSITION_20_50_VISIBLE,69134.0,27.753576,0,Medium — visible impressions with position 20–50.,Could be wrong if the position reflects a temp...
3,4,content_3df3f32f3fd58dea,REFRESH_REVIEW,POSITION_20_50_VISIBLE,68216.0,24.899933,10,Medium — visible impressions with position 20–50.,Could be wrong if the position reflects a temp...
4,5,content_b51957d7f4abe47e,REFRESH_REVIEW,POSITION_20_50_VISIBLE,57558.0,27.816678,0,Medium — visible impressions with position 20–50.,Could be wrong if the position reflects a temp...
5,6,content_bdf60c86117079be,REFRESH_REVIEW,POSITION_20_50_VISIBLE,51346.0,33.418919,10,Medium — visible impressions with position 20–50.,Could be wrong if the position reflects a temp...
6,7,content_0aaa197051f58d6f,REFRESH_REVIEW,POSITION_20_50_VISIBLE,49903.0,34.694009,10,Medium — visible impressions with position 20–50.,Could be wrong if the position reflects a temp...
7,8,content_a3a1317f7c2bc3dd,REFRESH_REVIEW,POSITION_20_50_VISIBLE,47036.0,29.355751,0,Medium — visible impressions with position 20–50.,Could be wrong if the position reflects a temp...
8,9,content_87b9c790d43001dc,REFRESH_REVIEW,POSITION_20_50_VISIBLE,46002.0,30.531879,0,Medium — visible impressions with position 20–50.,Could be wrong if the position reflects a temp...
9,10,content_a3b985d3ee8a219b,REFRESH_REVIEW,POSITION_20_50_VISIBLE,45744.0,24.343509,0,Medium — visible impressions with position 20–50.,Could be wrong if the position reflects a temp...


## Weak Picks + Leakage Check

The baseline is intentionally simple, so some selections may be weak.

Weak picks are reviewed by checking whether the rule is relying on a signal that is too weak, missing important context, or using information that would not have been available at the February 28 decision point.

The baseline must not use March outcomes, target labels, future-window information, or private client details.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Weak-pick review

print("LOW-CONFIDENCE / WEAK-PICK REVIEW")
weak_picks = baseline_queue[
    baseline_queue["reason_code"].isin([
        "POSITION_20_50_LOW_VISIBILITY",
        "OTHER"
    ])
].head(10).copy()

display(
    weak_picks[
        [
            "rank",
            "content_hash_id",
            "score",
            "action",
            "reason_code",
            "feb_impressions",
            "feb_avg_position",
            "search_volume",
            "content_type"
        ]
    ]
)

LOW-CONFIDENCE / WEAK-PICK REVIEW


,rank,content_hash_id,score,action,reason_code,feb_impressions,feb_avg_position,search_volume,content_type
26963,26964,content_ecf275d0ecdd31b9,3,REVIEW,POSITION_20_50_LOW_VISIBILITY,99.0,21.065935,30,keyword article
26964,26965,content_945e6d0a0b3f2e8b,3,REVIEW,POSITION_20_50_LOW_VISIBILITY,99.0,24.722722,20,keyword article
26965,26966,content_9741a96747ea7f95,3,REVIEW,POSITION_20_50_LOW_VISIBILITY,99.0,31.232684,30,keyword article
26966,26967,content_12415a5ac1570217,3,REVIEW,POSITION_20_50_LOW_VISIBILITY,99.0,23.507357,0,keyword article
26967,26968,content_197b2397bf5bb7b5,3,REVIEW,POSITION_20_50_LOW_VISIBILITY,99.0,41.550019,2400,keyword article
26968,26969,content_e2c54dc673738e71,3,REVIEW,POSITION_20_50_LOW_VISIBILITY,99.0,30.643222,260,keyword article
26969,26970,content_19516fa20c6d08d3,3,REVIEW,POSITION_20_50_LOW_VISIBILITY,99.0,29.297006,70,keyword article
26970,26971,content_380a32c38f33e4cc,3,REVIEW,POSITION_20_50_LOW_VISIBILITY,99.0,35.095111,0,keyword article
26971,26972,content_41f2230ab6d362e5,3,REVIEW,POSITION_20_50_LOW_VISIBILITY,99.0,49.748179,0,keyword article
26972,26973,content_9aad561f871a2941,3,REVIEW,POSITION_20_50_LOW_VISIBILITY,99.0,25.620238,0,keyword article


In [16]:
# Leakage checks

print("LEAKAGE CHECK")

# Features used by the baseline
baseline_features = [
    "feb_impressions",
    "feb_avg_position",
    "search_volume",
    "content_type"
]

print("Features used by baseline:")
print(baseline_features)

# Check that March outcome / target columns are not present
future_or_label_columns = [
    "march_impressions",
    "march_clicks",
    "march_ctr",
    "target_declining",
    "leaked_feature"
]

found_future_columns = [
    col for col in future_or_label_columns
    if col in queue.columns
]

print("\nFuture/label columns found in queue:", found_future_columns)

if len(found_future_columns) == 0:
    print("PASS — No March outcome or target-derived columns are used.")

# Check that the scoring calculation only uses February/pre-decision fields
score_check = (
    (queue["score"] ==
     queue["position_20_50_signal"] * 3 +
     queue["position_10_20_signal"] * 2 +
     queue["visibility_signal"])
).all()

print("\nScore calculation check:", score_check)

if score_check:
    print("PASS — Score is calculated only from the defined baseline signals.")

LEAKAGE CHECK
Features used by baseline:
['feb_impressions', 'feb_avg_position', 'search_volume', 'content_type']

Future/label columns found in queue: []
PASS — No March outcome or target-derived columns are used.

Score calculation check: True
PASS — Score is calculated only from the defined baseline signals.


### Leakage Check Result

The baseline score uses only February/pre-decision signals:

- February impressions
- February average search position
- search volume
- content type

March outcomes and target-derived fields are not used in the scoring rule.

The earlier staleness signal was also excluded because its observed values did not provide a valid pre-decision staleness measure.

Therefore, this baseline is designed as a pre-March action queue rather than a model trained on the future outcome.

## Final Baseline Takeaway

The baseline prioritizes content with measurable February search visibility and average search positions between 20 and 50 for refresh review.

The signal review showed that search position provided a useful directional relationship with February impressions, while the available staleness field was not suitable for this decision point.

The final queue contains 301,152 content items. The strongest action group contains 10,318 items marked `REFRESH_REVIEW`.

The baseline is intentionally simple and should be treated as a review-prioritization rule, not a prediction model. Its main limitation is that it uses only a small set of February signals and does not account for factors such as search intent, content quality, seasonality, or other business context.

The leakage check passed: no March outcome or target-derived fields were used in the score.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.